**5) ML Dataset Preparation**

In [ ]:
from pyspark.sql import functions as F

# I keep only rows where my target (SEVERITY) exists
df_ml = df.filter(F.col("SEVERITY").isNotNull())

# I standardize text columns I will use (safe cleanup)
for c in ["ARREST_BORO", "AGE_GROUP", "PERP_SEX", "PERP_RACE", "OFNS_DESC"]:
    if c in df_ml.columns:
        df_ml = df_ml.withColumn(c, F.upper(F.trim(F.col(c))))

# I also make sure my target is clean
df_ml = df_ml.withColumn("SEVERITY", F.upper(F.trim(F.col("SEVERITY"))))

print("Rows for ML:", df_ml.count())
df_ml.groupBy("SEVERITY").count().orderBy(F.desc("count")).show(truncate=False)

Rows for ML: 5986025
+-----------+-------+
|SEVERITY   |count  |
+-----------+-------+
|MISDEMEANOR|3866398|
|FELONY     |1767834|
|VIOLATION  |297792 |
|OTHER      |54001  |
+-----------+-------+



**CELL 20** Select feature columns (categorical + numeric)



In [ ]:
# I define my feature columns (categorical + numeric) based on the dataset columns that exist
candidate_categoricals = ["ARREST_BORO", "AGE_GROUP", "PERP_SEX", "PERP_RACE", "OFNS_DESC"]
candidate_numerics     = ["PD_CD", "KY_CD", "ARREST_PRECINCT", "JURISDICTION_CODE",
                          "X_COORD_CD", "Y_COORD_CD", "Latitude", "Longitude"]

cat_cols = [c for c in candidate_categoricals if c in df_ml.columns]
num_cols = [c for c in candidate_numerics if c in df_ml.columns]

print("Categorical features:", cat_cols)
print("Numeric features:", num_cols)

# I keep only the needed columns
keep_cols = ["SEVERITY"] + cat_cols + num_cols
df_ml = df_ml.select(*keep_cols)

df_ml.show(5, truncate=False)

Categorical features: ['ARREST_BORO', 'AGE_GROUP', 'PERP_SEX', 'PERP_RACE', 'OFNS_DESC']
Numeric features: ['PD_CD', 'KY_CD', 'ARREST_PRECINCT', 'JURISDICTION_CODE', 'X_COORD_CD', 'Y_COORD_CD', 'Latitude', 'Longitude']
+-----------+-----------+---------+--------+--------------+-----------------+-----+-----+---------------+-----------------+----------+----------+------------------+------------------+
|SEVERITY   |ARREST_BORO|AGE_GROUP|PERP_SEX|PERP_RACE     |OFNS_DESC        |PD_CD|KY_CD|ARREST_PRECINCT|JURISDICTION_CODE|X_COORD_CD|Y_COORD_CD|Latitude          |Longitude         |
+-----------+-----------+---------+--------+--------------+-----------------+-----+-----+---------------+-----------------+----------+----------+------------------+------------------+
|MISDEMEANOR|B          |18-24    |M       |BLACK         |DANGEROUS DRUGS  |567  |235  |44             |0                |1006490.0 |244533.0  |40.837842121000044|-73.91962775199994|
|MISDEMEANOR|B          |18-24    |M       |W

**CELL 21** Train/Test split + caching (big data best practice)



In [ ]:
from pyspark.storagelevel import StorageLevel

# I split my data into train and test for fair evaluation
train_df, test_df = df_ml.randomSplit([0.8, 0.2], seed=42)

# I cache for speed (big data best practice)
train_df = train_df.persist(StorageLevel.MEMORY_AND_DISK)
test_df  = test_df.persist(StorageLevel.MEMORY_AND_DISK)

print("Train:", train_df.count(), "Test:", test_df.count())

Train: 4788218 Test: 1197807


**6) Feature Pipeline (Encoding + Scaling)**

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler

LABEL_COL = "SEVERITY"

# I create my label indexer (multiclass)
label_indexer = StringIndexer(
    inputCol=LABEL_COL,
    outputCol="label",
    handleInvalid="keep"
)

# I index + one-hot encode categoricals
indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in cat_cols
]

encoder = OneHotEncoder(
    inputCols=[f"{c}_idx" for c in cat_cols],
    outputCols=[f"{c}_ohe" for c in cat_cols],
    handleInvalid="keep"
)

# I assemble numeric + encoded features into one vector
assembler_inputs = [f"{c}_ohe" for c in cat_cols] + num_cols

assembler = VectorAssembler(
    inputCols=assembler_inputs,
    outputCol="features_raw",
    handleInvalid="keep"
)

# I scale features (helps LR/SVM/MLP)
scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withStd=True,
    withMean=False
)

# I fit the shared pipeline ONCE on train (fairness + speed)
feat_pipe = Pipeline(stages=[label_indexer] + indexers + [encoder, assembler, scaler]).fit(train_df)

train_ready = feat_pipe.transform(train_df).select("label", "features")
test_ready  = feat_pipe.transform(test_df).select("label", "features")

print("Prepared train rows:", train_ready.count(), "Prepared test rows:", test_ready.count())
train_ready.show(3, truncate=False)

Prepared train rows: 4788218 Prepared test rows: 1197807
+-----+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|label|features                                                                                                                                                                                                                       |
+-----+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|1.0  |(197,[2,7,86,89,98,189,190,191,192,193,194,195,196],[2.3805691760696672,2.3120345345313744,2.670022823005776,2.0008815451476267,2.54245191864113,NaN,NaN,1.2202481144957833,NaN,49.792267197794324,1.6041012475572365,NaN,NaN])|
|1.0  |(197,[2,

**7) Model Training Setup (4 models required)**

**CELL 23** Define evaluators + define 4 models (LR, DT, SVM, Deep Learning)



In [ ]:
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, LinearSVC, OneVsRest, MultilayerPerceptronClassifier, RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# I define my evaluators
eval_acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
eval_f1  = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
eval_wp  = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedPrecision")
eval_wr  = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedRecall")

# I count number of classes for MLP
num_classes = int(train_ready.select("label").distinct().count())
print("Number of classes:", num_classes)

models = {}

# 1) Logistic Regression
models["Logistic Regression"] = LogisticRegression(
    featuresCol="features", labelCol="label",
    maxIter=50, regParam=0.0, elasticNetParam=0.0
)

# 2) Decision Tree
models["Decision Tree"] = DecisionTreeClassifier(
    featuresCol="features", labelCol="label",
    maxDepth=10, minInstancesPerNode=50
)

# 3) SVM (multiclass using OneVsRest + LinearSVC)
svm_binary = LinearSVC(featuresCol="features", labelCol="label", maxIter=50, regParam=0.1)
models["SVM (OneVsRest LinearSVC)"] = OneVsRest(classifier=svm_binary, labelCol="label", featuresCol="features")

# 4) Deep Learning (Multilayer Perceptron)
# I set a simple architecture: input -> hidden -> hidden -> output
input_size = train_ready.select("features").first()["features"].size
layers = [input_size, 64, 32, num_classes]
models["Deep Learning (MLP)"] = MultilayerPerceptronClassifier(
    featuresCol="features", labelCol="label",
    layers=layers, maxIter=50, seed=42, blockSize=256
)

print("MLP layers:", layers)

Number of classes: 4
MLP layers: [197, 64, 32, 4]


**8) Train All 4 Models + Evaluate + Comparison Table**

**CELL 24** Train and evaluate all models + build comparison table


In [ ]:
from pyspark.sql import Row

results = []

for name, model in models.items():
    print("\n==============================")
    print("Training:", name)
    print("==============================")

    fitted = model.fit(train_ready)
    preds  = fitted.transform(test_ready).cache()

    acc = float(eval_acc.evaluate(preds))
    f1  = float(eval_f1.evaluate(preds))
    wp  = float(eval_wp.evaluate(preds))
    wr  = float(eval_wr.evaluate(preds))

    print(f"{name} -> Accuracy: {acc:.4f} | F1: {f1:.4f} | W-Precision: {wp:.4f} | W-Recall: {wr:.4f}")

    results.append(Row(Model=name, Accuracy=acc, F1=f1, WeightedPrecision=wp, WeightedRecall=wr))

    preds.unpersist()

results_df = spark.createDataFrame(results).orderBy(F.desc("F1"))
results_df.show(truncate=False)


Training: Logistic Regression


Py4JJavaError: An error occurred while calling o1224.fit.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 123.0 failed 1 times, most recent failure: Lost task 0.0 in stage 123.0 (TID 2853) (6e5f65dbc397 executor driver): java.lang.RuntimeException: Vector values MUST NOT be NaN or Infinity, but got (197,[2,7,86,89,98,189,190,191,192,193,194,195,196],[2.3805691760696672,2.3120345345313744,2.670022823005776,2.0008815451476267,2.54245191864113,NaN,NaN,1.2202481144957833,NaN,49.792267197794324,1.6041012475572365,NaN,NaN])
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.project_doConsume_0$(Unknown Source)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.processNext(Unknown Source)
	at org.apache.spark.sql.execution.BufferedRowIterator.hasNext(BufferedRowIterator.java:43)
	at org.apache.spark.sql.execution.WholeStageCodegenEvaluatorFactory$WholeStageCodegenPartit

**9) Numeric Data Quality Check + Fix (important for stability)**

**CELL 25** Check NaN/Infinity in numeric columns


In [ ]:
from pyspark.sql import functions as F

# I quickly check NaN / Infinity in numeric columns
checks = []
for c in num_cols:
    checks.append(
        F.sum(F.when(F.isnan(F.col(c)) | F.col(c).isin(float("inf"), float("-inf")), 1).otherwise(0)).alias(c)
    )

nan_inf_counts = df_ml.select(checks)
nan_inf_counts.show(truncate=False)

+-----+-----+---------------+-----------------+----------+----------+--------+---------+
|PD_CD|KY_CD|ARREST_PRECINCT|JURISDICTION_CODE|X_COORD_CD|Y_COORD_CD|Latitude|Longitude|
+-----+-----+---------------+-----------------+----------+----------+--------+---------+
|0    |0    |0              |0                |0         |0         |0       |0        |
+-----+-----+---------------+-----------------+----------+----------+--------+---------+



**CELL 26** Replace NaN/Inf and fill missing numeric values with medians



In [ ]:
from pyspark.sql import functions as F

df_clean = df_ml

# 1) I replace NaN/Infinity in numeric columns with NULL
for c in num_cols:
    df_clean = df_clean.withColumn(
        c,
        F.when(F.col(c).isin(float("inf"), float("-inf")), None)
         .when(F.isnan(F.col(c)), None)
         .otherwise(F.col(c))
    )

# 2) I fill NULL numeric values with the median (robust) or 0 if you prefer
# Median in Spark: approxQuantile
fill_map = {}
for c in num_cols:
    med = df_clean.approxQuantile(c, [0.5], 0.01)[0]  # 1% relative error
    if med is None:
        med = 0.0
    fill_map[c] = float(med)

df_clean = df_clean.fillna(fill_map)

print("Numeric fill values (medians):")
for k,v in fill_map.items():
    print(k, "->", v)

Numeric fill values (medians):
PD_CD -> 503.0
KY_CD -> 341.0
ARREST_PRECINCT -> 60.0
JURISDICTION_CODE -> 0.0
X_COORD_CD -> 1004950.0
Y_COORD_CD -> 208800.0
Latitude -> 40.74088057900008
Longitude -> -73.92539159499995


**10) Rebuild Train/Test + Feature Pipeline After Cleaning**